# Local RAG System

This notebook demonstrates a local Retrieval-Augmented Generation (RAG) system using **LangChain**, **Ollama**, **Llama 3**, **Nomic Embed Text**, and **ChromaDB**.

### Notebook Architecture & Execution Flow
Below is a detailed explanation of what each section in this notebook accomplishes:

1. **Imports & Setup**: Loads all the essential libraries required for the pipeline, including LangChain components, HuggingFace embeddings, ChromaDB, and Ollama.
2. **Configuration**: Defines the paths for the input documents (`data/`) and the persistent vector database (`chroma_db/`).
3. **Step 1: Data Ingestion**: Loads the raw PDF and Text files from the data directory. It then uses a `RecursiveCharacterTextSplitter` to break down large documents into smaller, overlapping chunks (1000 characters each) to preserve context, and assigns a unique ID to every chunk.
4. **Step 2: Store Embeddings in ChromaDB**: Converts the text chunks into numerical vectors (embeddings) using the `all-MiniLM-L6-v2` model. These vectors are stored persistently in ChromaDB, allowing the system to quickly find semantically similar text during a query.
5. **Step 3: Set Up RAG Query Chain**: Configures the local **Llama 3** model (via Ollama) and ties it together with the ChromaDB retriever. It establishes a strict prompt template forcing the LLM to ground its answers *only* in the retrieved context.
6. **Step 4: Ask Questions**: Tests the pipeline with various dynamic domain questions (e.g., about Databases, Classes, Data Centers). It prints the context-aware answer and explicitly lists the source chunks used to generate it.
7. **Step 5: System Metrics Report**: Dynamically calculates and prints a comprehensive overview of the pipeline's configuration, including chunking profiles, embedding dimensions, and model setups.

### Notebook Architecture & Execution Flow
Below is a detailed explanation of what each section in this notebook accomplishes:

1. **Imports & Setup**: Loads all the essential libraries required for the pipeline, including LangChain components, HuggingFace embeddings, ChromaDB, and Ollama.
2. **Configuration**: Defines the paths for the input documents (`data/`) and the persistent vector database (`chroma_db/`).
3. **Step 1: Data Ingestion**: Loads the raw PDF and Text files from the data directory. It then uses a `RecursiveCharacterTextSplitter` to break down large documents into smaller, overlapping chunks (1000 characters each) to preserve context, and assigns a unique ID to every chunk.
4. **Step 2: Store Embeddings in ChromaDB**: Converts the text chunks into numerical vectors (embeddings) using the `all-MiniLM-L6-v2` model. These vectors are stored persistently in ChromaDB, allowing the system to quickly find semantically similar text during a query.
5. **Step 3: Set Up RAG Query Chain**: Configures the local **Llama 3** model (via Ollama) and ties it together with the ChromaDB retriever. It establishes a strict prompt template forcing the LLM to ground its answers *only* in the retrieved context.
6. **Step 4: Ask Questions**: Tests the pipeline with various dynamic domain questions (e.g., about Databases, Classes, Data Centers). It prints the context-aware answer and explicitly lists the source chunks used to generate it.
7. **Step 5: System Metrics Report**: Dynamically calculates and prints a comprehensive overview of the pipeline's configuration, including chunking profiles, embedding dimensions, and model setups.

### Notebook Architecture & Execution Flow
Below is a detailed explanation of what each section in this notebook accomplishes:

1. **Imports & Setup**: Loads all the essential libraries required for the pipeline, including LangChain components, HuggingFace embeddings, ChromaDB, and Ollama.
2. **Configuration**: Defines the paths for the input documents (`data/`) and the persistent vector database (`chroma_db/`).
3. **Step 1: Data Ingestion**: Loads the raw PDF and Text files from the data directory. It then uses a `RecursiveCharacterTextSplitter` to break down large documents into smaller, overlapping chunks (1000 characters each) to preserve context, and assigns a unique ID to every chunk.
4. **Step 2: Store Embeddings in ChromaDB**: Converts the text chunks into numerical vectors (embeddings) using the `all-MiniLM-L6-v2` model. These vectors are stored persistently in ChromaDB, allowing the system to quickly find semantically similar text during a query.
5. **Step 3: Set Up RAG Query Chain**: Configures the local **Llama 3** model (via Ollama) and ties it together with the ChromaDB retriever. It establishes a strict prompt template forcing the LLM to ground its answers *only* in the retrieved context.
6. **Step 4: Ask Questions**: Tests the pipeline with various dynamic domain questions (e.g., about Databases, Classes, Data Centers). It prints the context-aware answer and explicitly lists the source chunks used to generate it.
7. **Step 5: System Metrics Report**: Dynamically calculates and prints a comprehensive overview of the pipeline's configuration, including chunking profiles, embedding dimensions, and model setups.

In [ ]:
import os
import shutil
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
import warnings
warnings.filterwarnings('ignore')

C:\Users\nipun\AppData\Local\Temp\ipykernel_64280\2413593509.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
C:\Users\nipun\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Configuration
Before running this, make sure you have added your `.pdf` and `.txt` files into the `data/` folder.

In [2]:
DATA_PATH = "data"
CHROMA_PATH = "chroma_db"

### Step 1: Data Ingestion

In [3]:
def load_documents():
    print(f"Loading documents from {DATA_PATH}...")
    
    # Load PDFs (each page becomes a document)
    pdf_loader = DirectoryLoader(
        DATA_PATH,
        glob="**/*.pdf",
        loader_cls=PyPDFLoader,
        recursive=True,
        show_progress=True
    )
    
    # Load TXT files
    txt_loader = DirectoryLoader(
        DATA_PATH,
        glob="**/*.txt",
        loader_cls=TextLoader,
        recursive=True,
        show_progress=True
    )

    documents = []

    try:
        pdf_docs = pdf_loader.load()
        print(f"  Loaded {len(pdf_docs)} PDF pages.")
        documents.extend(pdf_docs)
    except Exception as e:
        print(f"  Warning loading PDFs: {e}")

    try:
        txt_docs = txt_loader.load()
        print(f"  Loaded {len(txt_docs)} TXT documents.")
        documents.extend(txt_docs)
    except Exception as e:
        print(f"  Warning loading TXTs: {e}")

    # Show a breakdown of loaded files
    sources = set(doc.metadata.get('source', 'unknown') for doc in documents)
    print(f"\nTotal chunks loaded: {len(documents)} from {len(sources)} file(s):")
    for s in sorted(sources):
        print(f"  - {os.path.basename(s)}")

    return documents

def split_documents(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        is_separator_regex=False,
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Split into {len(chunks)} chunks.")
    return chunks

def calculate_chunk_ids(chunks):
    last_page_id = None
    current_chunk_index = 0
    
    for chunk in chunks:
        source = chunk.metadata.get("source")
        page = chunk.metadata.get("page")
        
        if page is None:
            current_page_id = f"{source}"
        else:
            current_page_id = f"{source}:{page}"
            
        if current_page_id == last_page_id:
            current_chunk_index += 1
        else:
            current_chunk_index = 0
            
        chunk_id = f"{current_page_id}:{current_chunk_index}"
        last_page_id = current_page_id
        chunk.metadata["id"] = chunk_id
        
    return chunks

### Step 2: Store Embeddings in ChromaDB

In [4]:
def add_to_chroma(chunks, clear_existing=False):
    if clear_existing:
        print("Wiping existing database folder to prevent dimension mismatch...")
        if os.path.exists(CHROMA_PATH):
            try:
                shutil.rmtree(CHROMA_PATH)
                print("Database folder deleted successfully.")
            except PermissionError as e:
                print("\n" + "="*60)
                print("ERROR: Cannot delete 'chroma_db' folder because it is locked by Jupyter/VS Code.")
                print("Please RESTART the Jupyter Kernel in your notebook to unlock the files, then run again.")
                print("="*60 + "\n")
                return

    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    db = Chroma(persist_directory=CHROMA_PATH, embedding_function=embeddings)

    chunks_with_ids = calculate_chunk_ids(chunks)
    existing_items = db.get(include=[])
    existing_ids = set(existing_items["ids"])
    print(f"Existing chunks in DB after clear: {len(existing_ids)}")

    new_chunks = [chunk for chunk in chunks_with_ids if chunk.metadata["id"] not in existing_ids]

    if new_chunks:
        print(f"Adding {len(new_chunks)} new chunks to ChromaDB...")
        batch_size = 50
        total_batches = (len(new_chunks) - 1) // batch_size + 1
        for i in range(0, len(new_chunks), batch_size):
            batch = new_chunks[i:i+batch_size]
            batch_ids = [chunk.metadata["id"] for chunk in batch]
            db.add_documents(batch, ids=batch_ids)
            print(f"  Batch {i//batch_size + 1}/{total_batches} done ({i+len(batch)}/{len(new_chunks)} chunks)")
        print(f"\nAll {len(new_chunks)} chunks added to ChromaDB successfully!")
    else:
        print("No new documents to add. DB is up to date.")

# --- Run Ingestion ---
# Set clear_existing=True to wipe and re-embed everything fresh
# Set clear_existing=False to only add new files (skip already embedded ones)
documents = load_documents()
chunks = split_documents(documents)
add_to_chroma(chunks, clear_existing=True)

Loading documents from data...


100%|██████████| 4/4 [00:27<00:00,  6.97s/it]


  Loaded 1881 PDF pages.


0it [00:00, ?it/s]


  Loaded 0 TXT documents.

Total chunks loaded: 1881 from 4 file(s):
  - Artificial Intelligence.pdf
  - ArtificialIntelligenceAIBasedData.pdf
  - OOP_book.pdf
  - database-concepts.pdf
Split into 4886 chunks.
Wiping existing database folder to prevent dimension mismatch...
Database folder deleted successfully.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8797.93it/s]


Existing chunks in DB after clear: 0
Adding 4886 new chunks to ChromaDB...
  Batch 1/98 done (50/4886 chunks)
  Batch 2/98 done (100/4886 chunks)
  Batch 3/98 done (150/4886 chunks)
  Batch 4/98 done (200/4886 chunks)
  Batch 5/98 done (250/4886 chunks)
  Batch 6/98 done (300/4886 chunks)
  Batch 7/98 done (350/4886 chunks)
  Batch 8/98 done (400/4886 chunks)
  Batch 9/98 done (450/4886 chunks)
  Batch 10/98 done (500/4886 chunks)
  Batch 11/98 done (550/4886 chunks)
  Batch 12/98 done (600/4886 chunks)
  Batch 13/98 done (650/4886 chunks)
  Batch 14/98 done (700/4886 chunks)
  Batch 15/98 done (750/4886 chunks)
  Batch 16/98 done (800/4886 chunks)
  Batch 17/98 done (850/4886 chunks)
  Batch 18/98 done (900/4886 chunks)
  Batch 19/98 done (950/4886 chunks)
  Batch 20/98 done (1000/4886 chunks)
  Batch 21/98 done (1050/4886 chunks)
  Batch 22/98 done (1100/4886 chunks)
  Batch 23/98 done (1150/4886 chunks)
  Batch 24/98 done (1200/4886 chunks)
  Batch 25/98 done (1250/4886 chunks)
  Ba

### Step 3: Set Up RAG Query Chain

In [9]:
PROMPT_TEMPLATE = """
Answer the question comprehensively and in detail based only on the following context:\n",
{context}

---

Question: {input}
"""

# 1. Initialize the components ONCE globally
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = Chroma(persist_directory=CHROMA_PATH, embedding_function=embeddings)
retriever = db.as_retriever(search_kwargs={"k": 7})

# You can change "llama3" to a lighter model like "llama3.2" here if needed
llm = Ollama(model="llama3")
prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

document_chain = create_stuff_documents_chain(llm, prompt)
GLOBAL_RAG_CHAIN = create_retrieval_chain(retriever, document_chain)

# 2. Reuse the global chain in your query function
def query_rag(query_text: str):
    response = GLOBAL_RAG_CHAIN.invoke({"input": query_text})
    return response["answer"], response["context"]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9293.41it/s]


### Step 4: Ask Questions

In [10]:
query1 = "What is Database?"

answer, context = query_rag(query1)
print("=== Answer ===")
print(answer)
print("\n=== Sources ===")
for doc in context:
    print(f"- {doc.metadata.get('id', 'Unknown')}")

=== Answer ===
Based on the provided context, a database can be defined as:

"A database is an organized collection of information treated as a unit. The purpose of a database is to collect, store, and retrieve related information for use by database applications."

In other words, a database is a system that stores and manages data in a way that allows it to be easily accessed, manipulated, and retrieved by various applications or users. A database can be thought of as an electronic filing cabinet that organizes and structures information in a way that makes it useful for a particular purpose.

The main characteristics of a database include:

1. Organized collection of information: A database stores data in a structured format, making it easy to find and use the information.
2. Unitary concept: A database is treated as a single entity, with all its components (data, metadata, and applications) working together seamlessly.
3. Purpose-driven: The primary purpose of a database is to supp

In [11]:
query2 = "How database works and its properties?"

answer, context = query_rag(query2)
print("=== Answer ===")
print(answer)
print("\n=== Sources ===")
for doc in context:
    print(f"- {doc.metadata.get('id', 'Unknown')}")

=== Answer ===
Based on the provided context, a database is defined as an organized collection of information treated as a unit. Its purpose is to collect, store, and retrieve related information for use by database applications.

A Database Management System (DBMS) is software that controls the storage, organization, and retrieval of data. It typically consists of:

* Kernel code: manages memory and storage for the DBMS
* Repository of metadata: usually called a data dictionary
* Query language: enables applications to access the data

The main function of a DBMS is to provide a way to interact with a database. This interaction includes storing, retrieving, and manipulating data.

There are different types of DBMS, including:

1. Hierarchical: organizes data in a tree structure, where each parent record has one or more child records
2. Network: similar to hierarchical, but records have a many-to-many relationship rather than a one-to-many

The relational model is the most widely accep

In [14]:
query3 = "What is Class?"

answer, context = query_rag(query3)
print("=== Answer ===")
print(answer)
print("\n=== Sources ===")
for doc in context:
    print(f"- {doc.metadata.get('id', 'Unknown')}")

=== Answer ===
Based on the provided context, I will answer the question comprehensively and in detail.

A class is a plan or blueprint that defines what data and functions an object of that class should have. It serves as a template for creating objects with similar characteristics. A class does not create any objects; it only specifies what members (data and functions) will be included in objects of that class.

Think of classes like categories or groups in real life. For example, "rock musician" is a class that includes specific individuals such as Prince, Sting, and Madonna who possess certain characteristics. Similarly, in the context of programming, a class is a description of a number of similar objects.

In object-oriented programming (OOP), an object is often referred to as an "instance" of a class. This means that each object created from a particular class is an instance of that class, with its own set of characteristics and behaviors defined by the class.

To illustrate thi

In [15]:
query4 = "What is DataCenters?"

answer, context = query_rag(query4)
print("=== Answer ===")
print(answer)
print("\n=== Sources ===")
for doc in context:
    print(f"- {doc.metadata.get('id', 'Unknown')}")

=== Answer ===
Based on the provided context, a Data Center can be defined as a centralized location that houses and manages large amounts of digital information, processing power, and storage capacity. It serves as a hub for managing and analyzing massive datasets, often referred to as "Big Data," and supports various digital applications and services.

Data Centers are designed to provide high levels of scalability, reliability, and performance to meet the demands of modern computing and data-intensive applications. They typically consist of a complex infrastructure, including servers, storage devices, networking equipment, and power supplies, all working together to process and manage vast amounts of data.

In the context of AI data center networking, Data Centers are critical hubs for deploying artificial intelligence (AI) and machine learning (ML) technologies to optimize network operations, traffic routing, load balancing, predictive maintenance, and security monitoring. These ad

### Step 5: System Metrics Report

In [3]:
try:
    doc_count = len(documents)
except NameError:
    doc_count = "N/A (Run Step 1 to populate)"

try:
    chunk_count = len(chunks)
except NameError:
    chunk_count = "N/A (Run Step 1 to populate)"

try:
    chroma_path = CHROMA_PATH
except NameError:
    chroma_path = "chroma_db"

print("="*40)
print("       SYSTEM METRICS REPORT")
print("="*40)
print("1. Chunking Profile:")
print("   - Strategy: RecursiveCharacterTextSplitter")
print("   - Chunk Size: 1000 characters")
print("   - Chunk Overlap: 200 characters")
print(f"   - Total Documents Processed: {doc_count}")
print(f"   - Total Chunks Generated: {chunk_count}")
print("\n2. Text Embedding Setup:")
print("   - Model: HuggingFace (all-MiniLM-L6-v2)")
print("   - Dimensions: 384")
print("\n3. Vector Store Tool:")
print("   - Database: ChromaDB")
print(f"   - Persistence Directory: {chroma_path}")
print("\n4. Language Model Setup:")
print("   - Provider: Ollama")
print("   - Model: Llama 3")
print("   - Temperature: Default")
print("="*40)

       SYSTEM METRICS REPORT
1. Chunking Profile:
   - Strategy: RecursiveCharacterTextSplitter
   - Chunk Size: 1000 characters
   - Chunk Overlap: 200 characters
   - Total Documents Processed: N/A (Run Step 1 to populate)
   - Total Chunks Generated: N/A (Run Step 1 to populate)

2. Text Embedding Setup:
   - Model: HuggingFace (all-MiniLM-L6-v2)
   - Dimensions: 384

3. Vector Store Tool:
   - Database: ChromaDB
   - Persistence Directory: chroma_db

4. Language Model Setup:
   - Provider: Ollama
   - Model: Llama 3
   - Temperature: Default
